# AIGC Detection — Iteration v2

Second iteration of the AI-generated-image detector. Iteration **v1** lives in
`Qwen.ipynb` (kept as the documented baseline); this notebook is a copy of it
with the **v2 robustness recipe** layered in. What changed vs v1, and why:

| Lever | v1 | v2 | Motivation |
| :-- | :-- | :-- | :-- |
| **Train-time augmentation** | none | jpeg / blur / resize / noise / color / crop at random severity | The robustness eval's one real failure was additive **noise** (AIGC recall 0.47→0.22). v1 never saw a transformed image. |
| **Train source** | SID_Set only | SID_Set (streamed) **+** WildFake non-val subset | Closes the ~66% cross-dataset gap — v1 overfit SID's generators (VQDM 0%). |
| **AIGC score** | hard 1.0 / 0.0 | **soft** P(AIGC) via teacher-forced label likelihood | Deliverable wants a likelihood; lets the biased operating point be tuned. |
| **Eval holdout** | positional `take` / `skip` (fragile) | **content-hash** routing (`hash % K`) | Leak-proof across reshuffles; also dedups near-duplicates. |
| **Budget** | 500 steps, LR 5e-6, eff-batch 4 | ~2500 steps, LR 5e-6, eff-batch 8 | Absorbs the added data/augmentation; keeps the collapse-safe LR. |

See `key-takeaways.md` (2026-08-30) for the analysis behind these choices.
**Companion step:** run `pull_wildfake_train.py` first to populate
`wildfake_train/` (a larger, non-val WildFake set, guaranteed disjoint from the
`wildfake_balanced/` eval set).

In [ ]:
!pip install torch torchvision transformers datasets trl bitsandbytes scikit-learn tqdm modelscope addict simplejson sortedcontainers --break-system-packages

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from torchvision import transforms

import sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/TechJam



# stream SID trainset because our laptop is too small for it
dataset = load_dataset("saberzl/SID_Set", split="train", streaming=True)

# tested with buffer sizes: [100,500,1000,2500,5000,10000], 10000 seems to be upper bound for what works
shuffled_stream = dataset.shuffle(buffer_size=10000, seed=42)


In [ ]:
# ============================================================================
# === WILDFAKE TRAIN STREAM (ModelScope) -> HF IterableDataset ===============
# Option 2: stream WildFake straight from ModelScope as a SECOND train source,
# wrapped so it drops into the same machinery `sid_train` uses. Two things this
# cell guarantees so the mix stays legal + typed:
#   1. VAL-EXCLUSION (the brief forbids training on the reference benchmark).
#      The WildFake val subset = COCO val2017 reals + DALL-E "Advanced" AIGC.
#      _wfs_is_val() drops exactly those rows; everything else (fakes from the
#      other generators, non-COCO reals) survives -> pure generator diversity.
#   2. TYPE MATCH. MsDataset's streaming object is NOT a datasets.IterableDataset,
#      so `.filter/.map/interleave_datasets` can't touch it. We re-wrap it with
#      IterableDataset.from_generator yielding {image: PIL, label: int}, cast to
#      the SAME features as `sid_train` (image=Image(), label=int64). It then
#      interleaves 1:1 with the SID stream.
# Label convention matches SID_Set: 0 = Real, 1 = Synthetic (fake). Lazy: nothing
# is streamed until the trainer pulls from it. Falls back to None (SID-only) if
# modelscope / the stream is unreachable, mirroring the local-folder path.
# ============================================================================
import os
from datasets import IterableDataset as HFIterableDataset, Features, Value, Image as HFImage
from PIL import Image as _PILImage

# --- Where to stream from (same coords as the WildFake eval cell) -----------
WILDFAKE_STREAM_DATASET = "hy2628982280/WildFake"
WILDFAKE_STREAM_SUBSET  = "default"
WILDFAKE_STREAM_SPLIT   = "train"
# Streamed rows may expose only a relative Image_path ("./...") instead of decoded
# pixels (WildFake ships pixels inside multi-GB zips). If so, point this at the
# local Images root so paths resolve; None = assume the row carries an image.
WILDFAKE_STREAM_IMAGES_ROOT = None
USE_WILDFAKE_STREAM = True            # flip to False to skip this source entirely


def _wfs_truthy(v):
    """Coerce IsFake / IsAdvanced ('1'/'0'/1/0/'True'/'False') to bool."""
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y", "t"}
    return bool(v)


def _wfs_haystack(ex):
    """Lower-cased blob of the source-identifying fields, for substring tests."""
    return " ".join(
        str(ex.get(k, "")) for k in ("Architecture", "Category", "Generator", "Image_path")
    ).lower()


def _wfs_is_val(ex):
    """True iff this row belongs to the held-out WildFake benchmark and must be
    kept OUT of training: a COCO real, OR a DALL-E-Advanced fake."""
    hay = _wfs_haystack(ex)
    is_coco_real = (not _wfs_truthy(ex.get("IsFake"))) and "coco" in hay
    is_dalle_adv = (
        _wfs_truthy(ex.get("IsFake"))
        and "dalle" in hay
        and _wfs_truthy(ex.get("IsAdvanced"))
    )
    return is_coco_real or is_dalle_adv


def _wfs_image(ex):
    """PIL.Image (RGB) for a streamed row, or None if pixels are unresolvable."""
    for k in ("image", "Image", "img"):
        obj = ex.get(k)
        if isinstance(obj, _PILImage.Image):
            return obj.convert("RGB")
    path = ex.get("Image_path")
    if path:
        if WILDFAKE_STREAM_IMAGES_ROOT:
            path = os.path.join(WILDFAKE_STREAM_IMAGES_ROOT, str(path).lstrip("./"))
        try:
            return _PILImage.open(path).convert("RGB")
        except Exception:
            return None
    return None


def _wildfake_stream_gen():
    """Yield {image, label} for every WildFake train row that is NOT in the val
    benchmark and whose pixels resolve. Re-opens the stream on each epoch."""
    from modelscope.msdatasets import MsDataset
    stream = MsDataset.load(
        WILDFAKE_STREAM_DATASET, subset_name=WILDFAKE_STREAM_SUBSET,
        split=WILDFAKE_STREAM_SPLIT, use_streaming=True,
    )
    for ex in stream:
        if _wfs_is_val(ex):                       # anti-leak: skip the benchmark
            continue
        img = _wfs_image(ex)
        if img is None:                           # unresolved pixels -> skip
            continue
        yield {"image": img, "label": 1 if _wfs_truthy(ex.get("IsFake")) else 0}


# Same features as `sid_train` so interleave_datasets accepts both sources.
_wfs_features = Features({"image": HFImage(), "label": Value("int64")})

wildfake_stream = None
if USE_WILDFAKE_STREAM:
    try:
        wildfake_stream = HFIterableDataset.from_generator(
            _wildfake_stream_gen, features=_wfs_features,
        )
        # Light peek so schema / reachability surface here, not deep in training.
        _wfs_peek = next(iter(wildfake_stream))
        print("wildfake_stream OK -> keys:", list(_wfs_peek.keys()),
              "| first label:", _wfs_peek["label"],
              "| image:", _wfs_peek["image"].size)
    except Exception as e:
        wildfake_stream = None
        print(f"WARNING: wildfake_stream unavailable ({type(e).__name__}: {e}). "
              f"Training will fall back to SID (+ local WildFake folder) only.")

# To use it, interleave in the 'BUILD THE MIXED TRAIN SOURCE' cell, e.g.:
  train_source = interleave_datasets(
      [sid_train, wildfake_stream],
      probabilities=[1 - WILDFAKE_MIX_PROB, WILDFAKE_MIX_PROB],
      seed=SEED, stopping_strategy="all_exhausted")


In [ ]:
# ============================================================================
# === CONFIGURATION - ALL SETTINGS IN ONE PLACE (v2) ===
# ============================================================================

# --- Model Configuration ---
# image-text-to-text (vision-language) variant -- NOT the text-only "-Base".
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# --- Dataset Configuration ---
#TODO: REPLACE WITH YOUR OWN PATH
DATASET_PATH = ""

# --- Training Configuration (feel free to adjust!) ---
# v2 budget: keep the collapse-safe LR (5e-6) but spend MORE on learning --
# ~2500 steps at effective batch 8 = ~20k images seen, still << SID_Set's ~210k
# so the trainer never completes an epoch (no reshuffle-leak).
# SPEED (lever #2): the effective batch stays 8, but we now push REAL parallelism
# onto the GPU (per-device batch 4) instead of running 8 micro-batches serially.
# A 0.8B VLM leaves the A100 mostly idle at batch 1; 4 x accum 2 keeps the exact
# same LR / warmup / images-per-step -- only the wall-clock changes. Bump to 8 x 1
# if VRAM allows; on OOM drop back to 4 x 2 (or 2 x 4). Keep the PRODUCT = 8 so
# the tuned LR/step budget still holds.
TRAIN_BATCH_SIZE = 8                  # v2b: was 4. GPU RAM sat at 11/40GB -> use it
GRADIENT_ACCUMULATION_STEPS = 1      # v2b: was 2. 8 x 1 = effective 8 (unchanged)
WARMUP_STEPS = 25                    # v1: 4  -> scaled up with the longer run
MAX_STEPS = 2500                     # v1: 500
LEARNING_RATE = 5e-6                 # v1: 5e-6 (KEPT -- halving it just undoes the extra steps)
WEIGHT_DECAY = 0.025
LR_SCHEDULER_TYPE = "linear"
OPTIM = "adamw_8bit"                 # requires bitsandbytes
SEED = 189

# --- Train-time augmentation (v2 lever #1: fix the noise-robustness failure) --
# Applied to TRAIN images only (never to eval). Mirrors the robustness eval's
# transform family so the model learns invariance to the transforms it is scored
# under. Noise reaches 0.10 (the severity that crushed v1's AIGC recall).
AUGMENT_TRAIN   = True
AUG_PROB        = 0.85               # P(apply >=1 transform); rest stay pristine
AUG_MAX_OPS     = 3                  # up to this many transforms stacked per image
AUG_JPEG_RANGE  = (30, 95)           # JPEG quality
AUG_BLUR_RANGE  = (0.0, 2.0)         # Gaussian blur radius (px)
AUG_RESIZE_RANGE = (0.25, 1.0)       # downscale factor, then back up
AUG_NOISE_RANGE = (0.0, 0.10)        # Gaussian noise sigma (fraction of 255)
AUG_COLOR_RANGE = (0.7, 1.3)         # colour/brightness/contrast enhance factor
AUG_CROP_RANGE  = (0.7, 0.95)        # centre-crop keep fraction, then resize back
AUG_SEED        = 12345

# --- WildFake mix (v2 lever #2: generator diversity) ------------------------
# Interleave streamed SID_Set with a locally-pulled WildFake non-val subset
# (build it with pull_wildfake_train.py). WILDFAKE_MIX_PROB is the sampling
# weight given to WildFake in the interleave. If the folder is missing the
# notebook falls back to SID-only (prints a warning) so it still runs.
USE_WILDFAKE_MIX   = True
WILDFAKE_TRAIN_DIR = "wildfake_train"
WILDFAKE_MIX_PROB  = 0.5             # 0.5 = roughly balanced SID vs WildFake

# --- Evaluation Configuration ---
EVAL_MAX_NEW_TOKENS = 128            # only used by the (reference) greedy-generate path
N_EVAL = 100                         # held-out samples reserved from the SID stream
# v2 lever #4: content-hash holdout. A SID sample is routed to EVAL iff
# hash(image) % EVAL_HOLDOUT_MOD == 0; training keeps the complement. Stable
# across reshuffles/epochs (unlike v1's positional take/skip) and dedups near-
# duplicates. MOD=20 -> ~5% of the stream is eval-eligible.
EVAL_HOLDOUT_MOD = 20
# v2 lever #3: emit a SOFT AIGC probability (teacher-forced label likelihood)
# instead of a hard 1.0/0.0. Set False to fall back to greedy-generate + parse.
SOFT_SCORE = True

OUTPUT_DIR = "./aigc_detector_qwen_v2"

### Load base model & tokenizer

In [ ]:
# Load the VLM as an image-text-to-text model + its multimodal processor.
# If AutoModelForImageTextToText doesn't resolve for this checkpoint, try
# AutoModelForMultimodalLM (named on the model card) or add trust_remote_code=True.
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# SPEED: bf16 weights + PyTorch SDPA attention. On A100 + bf16, SDPA dispatches
# to the FlashAttention-2 kernel under the hood -- ~same speed as the standalone
# flash-attn package, with no wheel to match or compile (flash-attn ships no
# prebuilt FA2 wheel for Colab's torch 2.11). dtype=bfloat16 (not "auto") keeps
# weights half-precision for the fused kernel; pairs with bf16/tf32 in the config.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

# Downstream text helpers reference `tokenizer`; the processor bundles one.
tokenizer = processor.tokenizer
# VLM tokenizers may ship without a pad token; SFT batching needs one.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# ============================================================================
# === REAL-WORLD TRANSFORMS + TRAIN-TIME AUGMENTATION (v2 lever #1) ===
# ============================================================================
# Two users of the same six transforms:
#   * augment_image(...)  -- RANDOM severities, applied to the TRAIN stream so
#     the model learns invariance (esp. to noise, v1's one real failure).
#   * the deterministic _t_* helpers -- FIXED severities, reused by the
#     robustness-eval cell near the end to re-measure v2 the way v1 was measured.
# PIL in -> PIL out (the VLM collator consumes PIL images). Eval images scored
# "clean" are never passed through augment_image.
import io
import random as _random
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance


def _to_rgb(image):
    return image if image.mode == "RGB" else image.convert("RGB")


def _t_jpeg(image, quality):
    """Re-encode as JPEG at `quality` (block / ringing artifacts)."""
    buf = io.BytesIO()
    _to_rgb(image).save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def _t_blur(image, radius):
    """Gaussian blur (px radius)."""
    return _to_rgb(image).filter(ImageFilter.GaussianBlur(radius=float(radius)))


def _t_resize(image, factor):
    """Downscale by `factor`, then back to the original size (detail loss)."""
    image = _to_rgb(image)
    w, h = image.size
    small = image.resize((max(1, int(w * factor)), max(1, int(h * factor))), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)


def _t_noise(image, sigma, rng=np.random):
    """Additive Gaussian noise; `sigma` is a fraction of 255 (e.g. 0.02..0.10)."""
    arr = np.asarray(_to_rgb(image)).astype(np.float32)
    arr = arr + rng.normal(0.0, float(sigma) * 255.0, arr.shape)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def _t_color(image, factor):
    """Colour jitter: scale saturation, brightness and contrast by `factor`."""
    out = ImageEnhance.Color(_to_rgb(image)).enhance(float(factor))
    out = ImageEnhance.Brightness(out).enhance(float(factor))
    return ImageEnhance.Contrast(out).enhance(float(factor))


def _t_crop(image, keep):
    """Centre-crop to `keep` of each side, then resize back to the original."""
    image = _to_rgb(image)
    w, h = image.size
    cw, ch = int(w * keep), int(h * keep)
    left, top = (w - cw) // 2, (h - ch) // 2
    return image.crop((left, top, left + cw, top + ch)).resize((w, h), Image.BILINEAR)


def augment_image(image, rng=None):
    """Randomly stack 1..AUG_MAX_OPS transforms at random severities on a PIL
    image. Order is fixed+sensible (geometry -> blur -> colour -> noise -> jpeg,
    so jpeg sees the composited result, as in the wild). With prob (1 - AUG_PROB)
    the image is returned clean so the model still sees pristine examples."""
    if image is None:
        return image
    rng = rng or _random.Random()
    image = _to_rgb(image)
    if rng.random() > AUG_PROB:
        return image

    ops = rng.sample(["resize", "crop", "blur", "color", "noise", "jpeg"],
                     k=rng.randint(1, AUG_MAX_OPS))
    if "resize" in ops:
        image = _t_resize(image, rng.uniform(*AUG_RESIZE_RANGE))
    if "crop" in ops:
        image = _t_crop(image, rng.uniform(*AUG_CROP_RANGE))
    if "blur" in ops:
        image = _t_blur(image, rng.uniform(*AUG_BLUR_RANGE))
    if "color" in ops:
        image = _t_color(image, rng.uniform(*AUG_COLOR_RANGE))
    if "noise" in ops:
        image = _t_noise(image, rng.uniform(*AUG_NOISE_RANGE))
    if "jpeg" in ops:
        image = _t_jpeg(image, rng.randint(*AUG_JPEG_RANGE))
    return image


# One RNG for the whole training stream (reproducible given AUG_SEED).
_aug_rng = _random.Random(AUG_SEED)

---
### Prompt Construction

Each SID_Set example carries an `image` (PIL) and a `label` (int); the `label`
meaning is defined by the dataset card:

| label | category  | meaning                                        |
| :---: | :-------- | :--------------------------------------------- |
|   0   | Real      | authentic photograph                           |
|   1   | Synthetic | image fully generated by AI                    |
|   2   | Tampered  | real image with AI-manipulated / edited regions |

The model is a vision-language model, so the **real image** is fed alongside the
text as a separate content item (the processor inserts the actual image tokens);
we do **not** splice a `<image>` string into the prompt. The assistant answers
with the **raw integer label** in `\boxed{}` form (e.g. `\boxed{1}`), so no
letter / class-name mapping is needed and 3-class scoring is a direct
`parsed_int == gold_int` comparison.

* **`build_sid_prompt_text`** — the task text (user turn): states the task, lists
  the class options by their integer label, and ends on `Reasoning:` so the model
  continues from there.
* **`build_sid_eval_messages`** — inference messages (image inlined, no assistant
  turn) for a direct `processor.apply_chat_template(...)` call at eval time.
* **`build_sid_train_prompt` / `build_sid_train_completion`** — the training
  sample split into a `prompt` turn (image placeholder + task) and a `completion`
  turn (gold `\boxed{d}` answer). This prompt/completion format lets TRL's vision
  collator mask the prompt and train loss on the answer only
  (`completion_only_loss=True`); real pixels come from the `image` column.


In [ ]:
# ============================================================================
# === PROMPT BUILDERS FOR THE STREAMED SID_Set DATASET ===
# (adapted from build_mmlu_prompt / build_mmlu_sft_text in the fine-tuning
#  tutorial; zero-shot, multimodal)
# ============================================================================

# --- Label schema (from the SID_Set dataset card) --------------------------
# Integer label -> (short name, human-readable description). The model answers
# with the raw integer, so there is no option-letter column.
SID_LABELS = {
    0: ("Real",      "an authentic, unmodified photograph"),
    1: ("Synthetic", "an image fully generated by AI (e.g. a diffusion / GAN model)"),
    2: ("Tampered",  "a real image with AI-manipulated or edited regions"),
}


def build_sid_prompt_text() -> str:
    """Task text (user turn) for a single image.

    No `<image>` placeholder string: in the conversational VLM format the image
    is a separate content item and the processor inserts the real image tokens.
    The model answers with the category's integer label in \\boxed{} form,
    parsed back by `parse_label_from_boxed`.
    """
    options = "\n".join(
        f"{label}. {name} \u2014 {desc}"
        for label, (name, desc) in SID_LABELS.items()
    )
    return (
        "You are an expert image-forensics analyst detecting AI-generated images.\n"
        "Classify the image into exactly one category.\n"
        "1. First, give a brief analysis of visual cues such as textures, edges, "
        "lighting, reflections, anatomy, text, and compression artifacts.\n"
        "2. Then, output the final answer as the category number inside a LaTeX box, "
        "e.g., \\boxed{0}.\n\n"
        f"Options:\n{options}\n\n"
        "Reasoning:"  # <--- The model will start generating from here
    )


def build_sid_target(label: int) -> str:
    """Assistant turn (zero-shot): the gold answer as the raw label in \\boxed{}."""
    return f"\\boxed{{{int(label)}}}"


def _sid_user_content(image=None):
    """User content = an image item + the task text.

    `image=None` -> a bare `{"type": "image"}` placeholder (training: real pixels
    come from the dataset's `image` column). `image=<PIL>` -> the pixels inlined
    (eval: a direct `processor.apply_chat_template(...)` call).
    """
    image_item = {"type": "image"} if image is None else {"type": "image", "image": image}
    return [image_item, {"type": "text", "text": build_sid_prompt_text()}]


def build_sid_eval_messages(image):
    """Inference messages for one image (image inlined, no assistant turn)."""
    return [{"role": "user", "content": _sid_user_content(image)}]


def build_sid_train_prompt():
    """Training PROMPT turn (user): image placeholder + task text.

    We use the prompt/completion format (a `prompt` list + a `completion` list),
    NOT a single `messages` list, so TRL's vision collator can mask the prompt and
    compute loss ONLY on the completion (`completion_only_loss=True`). This is the
    answer-only-loss fix for VLMs -- TRL rejects `assistant_only_loss` for vision
    datasets ("Assistant-only loss is not yet supported for vision datasets").
    """
    return [{"role": "user", "content": _sid_user_content(None)}]


def build_sid_train_completion(label: int):
    """Training COMPLETION turn (assistant): the gold \\boxed{d} answer.

    Real pixels come from the dataset's `image` column, matched to the
    `{"type": "image"}` placeholder in the prompt by the VLM SFT collator.
    """
    return [{"role": "assistant", "content": [{"type": "text", "text": build_sid_target(label)}]}]


In [ ]:
# --- Sanity check: what the model actually sees -----------------------------
from itertools import islice

print("=== TASK TEXT (user turn) ===")
print(build_sid_prompt_text())

print("\n=== TRAIN PROMPT / COMPLETION (label=1 -> Synthetic) ===")
print("prompt:    ", build_sid_train_prompt())
print("completion:", build_sid_train_completion(1))

# Pull a couple of labels straight off the stream. islice avoids iterating the
# whole 210k-image split.
print("\n=== GOLD TARGETS FOR A FEW STREAMED SAMPLES ===")
for ex in islice(shuffled_stream, 2):
    print(f"label={ex['label']} -> {build_sid_target(ex['label'])}")


---
### Evaluation: baseline vs fine-tuned accuracy

To gauge whether fine-tuning helps, we score the model on a **held-out slice of
the stream** (never trained on) both **before** and **after** fine-tuning —
mirroring the tutorial's before/after pattern. Each image is fed through the
processor (real pixels), the model greedy-decodes an answer, and we parse the
`\boxed{d}` label. We report **3-class** accuracy (Real/Synthetic/Tampered) and
the **binary** deliverable metric (AIGC vs authentic).

Order matters: `.train()` mutates `model` in place, so the **baseline eval runs
before** the training cell and its numbers are captured in `baseline`.


#### 3-class vs binary accuracy — what's the difference?

`eval_sid_accuracy` (next cell) reports **two** accuracy numbers for the same
predictions. They differ only in how strict the "correct" test is.

**3-class accuracy** scores the model on the *raw SID_Set label* — did it name
the exact category out of three?

| label | category  |
| :---: | :-------- |
|   0   | Real      |
|   1   | Synthetic (fully AI-generated) |
|   2   | Tampered (real photo, AI-edited regions) |

A prediction is correct only if `pred == gold` exactly (`correct_3class`).
Confusing Synthetic with Tampered is **wrong** here.

**Binary accuracy** first collapses *both* the prediction and the gold label to
the two categories the hackathon actually grades (`collapse_to_binary`):

```
{1 Synthetic, 2 Tampered} -> "AIGC"
{0 Real}                   -> "authentic"
```

then scores `collapse(pred) == collapse(gold)` (`correct_binary`). Now a
Synthetic↔Tampered mix-up is **correct**, because both map to "AIGC".

**Why they differ.** Binary is always **≥** 3-class on the same predictions —
the only errors it forgives are Synthetic↔Tampered (both AIGC). It still
penalizes the mistakes that matter: calling a real photo AI-generated, or the
reverse. Example — gold = 2 (Tampered), model predicts 1 (Synthetic):

- 3-class: **wrong** (1 ≠ 2)
- binary: **right** (both collapse to "AIGC")

**Which one matters.** **Binary is the deliverable metric** — the brief grades
"AIGC vs authentic," so that's the number that reflects the real task. The
**3-class** number is a diagnostic: together with the per-class breakdown it
exposes the common failure mode where a model collapses to one majority class
(that class near 100%, the others near 0%). A large gap (high binary, low
3-class) means the model can tell AI from real but can't separate
fully-synthetic from tampered — which is fine for us, since we collapse them
anyway.


In [ ]:
# ============================================================================
# === EVAL HARNESS: parse \boxed{d}, collapse to binary, score accuracy ===
# (adapted from the tutorial's parse_choice_from_boxed / eval_mcq_accuracy +
#  eval_mcq_accuracy_majority). v2 adds a SOFT label-likelihood scorer.
# ============================================================================
import re
import pandas as pd
from collections import Counter
from sklearn.metrics import (f1_score, balanced_accuracy_score, confusion_matrix,
                             roc_auc_score, average_precision_score)


def parse_label_from_boxed(text):
    """Extract the predicted SID_Set integer label (0/1/2) from generated text.

    Mirrors the tutorial's parse_choice_from_boxed but targets digits: prefer a
    \boxed{d}; else fall back to the last standalone 0/1/2; else None.
    """
    if text is None:
        return None
    m = re.search(r"\\boxed\{\s*([0-2])\s*\}", text)
    if m:
        return int(m.group(1))
    digits = re.findall(r"\b([0-2])\b", text)
    if digits:
        return int(digits[-1])
    return None


def collapse_to_binary(label):
    """Collapse the 3-class label to the binary deliverable:
    {1 Synthetic, 2 Tampered} -> 'AIGC', {0 Real} -> 'authentic', None -> None."""
    if label is None:
        return None
    return "authentic" if int(label) == 0 else "AIGC"


@torch.no_grad()
def _sid_score_labels(model, processor, image):
    """SOFT scorer (v2 lever #3). Returns {0: p0, 1: p1, 2: p2}, the model's
    normalised likelihood of each label, computed by TEACHER-FORCING each
    candidate completion "\boxed{k}" after the (image + prompt) and softmax-ing
    the three sequence log-probs. Deterministic, no autoregressive sampling, and
    format-matched to training (the SFT completion is exactly "\boxed{d}").
    P(AIGC) = p1 + p2 is the deliverable likelihood; argmax is the hard label."""
    prompt = processor.apply_chat_template(
        build_sid_eval_messages(image), add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    prompt_len = prompt["input_ids"].shape[1]

    logps = []
    for k in (0, 1, 2):
        msgs = build_sid_eval_messages(image) + [
            {"role": "assistant",
             "content": [{"type": "text", "text": build_sid_target(k)}]}]
        full = processor.apply_chat_template(
            msgs, add_generation_prompt=False, tokenize=True,
            return_dict=True, return_tensors="pt").to(model.device)
        ids = full["input_ids"]
        logits = model(**full).logits[:, :-1, :].log_softmax(-1)
        tok_logp = logits.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)[0]
        # positions >= prompt_len-1 predict the completion tokens ids[prompt_len:]
        logps.append(float(tok_logp[prompt_len - 1:].sum()))

    m = max(logps)
    exps = [float(torch.exp(torch.tensor(lp - m))) for lp in logps]
    z = sum(exps)
    return {k: exps[k] / z for k in (0, 1, 2)}


def _sid_predict_proba(model, processor, image):
    """Soft prediction: (hard_label argmax, P(AIGC)=p1+p2, full prob dict)."""
    probs = _sid_score_labels(model, processor, image)
    pred = max(probs, key=probs.get)
    return pred, probs[1] + probs[2], probs


def _sid_predict(model, processor, image, max_new_tokens, n_votes, temperature, top_p):
    """Predict one image's label (0/1/2 or None) + a sample decoded string.

    n_votes == 1 -> greedy (do_sample=False). n_votes > 1 -> self-consistency:
    sample n_votes answers and take the majority (mode) of the parsed labels,
    dropping unparseable samples (adapted from eval_mcq_accuracy_majority).
    """
    inputs = processor.apply_chat_template(
        build_sid_eval_messages(image),
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    if n_votes == 1:
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        decoded = processor.decode(out[0][prompt_len:], skip_special_tokens=True)
        return parse_label_from_boxed(decoded), decoded

    # Majority voting: one batched call with num_return_sequences. If the VLM
    # can't expand image features that way, fall back to n_votes separate calls.
    try:
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=top_p, num_return_sequences=n_votes,
        )
        seqs = [out[i][prompt_len:] for i in range(out.shape[0])]
    except Exception:
        seqs = [
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                           temperature=temperature, top_p=top_p)[0][prompt_len:]
            for _ in range(n_votes)
        ]

    votes, last_decoded = [], ""
    for seq in seqs:
        last_decoded = processor.decode(seq, skip_special_tokens=True)
        v = parse_label_from_boxed(last_decoded)
        if v is not None:            # ignore unparseable samples
            votes.append(v)
    pred = Counter(votes).most_common(1)[0][0] if votes else None
    return pred, last_decoded


def print_confusion(cm, cm_labels, title=""):
    """Pretty-print a small confusion matrix (rows=gold, cols=pred)."""
    names = {0: "Real", 1: "Synth", 2: "Tamp", -1: "None"}
    print((title + "  (rows=gold, cols=pred)").strip())
    print("        " + "  ".join(f"{names[c]:>5s}" for c in cm_labels))
    for i, gl in enumerate(cm_labels):
        if gl == -1:                 # gold is never the -1 "unparsed" bucket
            continue
        row = "  ".join(f"{cm[i][j]:5d}" for j in range(len(cm_labels)))
        print(f"  {names[gl]:>5s} {row}")


@torch.no_grad()
def eval_sid_accuracy(model, processor, eval_samples,
                      max_new_tokens=EVAL_MAX_NEW_TOKENS,
                      n_votes=1, temperature=0.7, top_p=0.9, soft=SOFT_SCORE):
    """Score held-out images: 3-class + binary acc, macro-F1, balanced acc, confusion.

    `soft=True` (default in v2) uses the deterministic teacher-forced label
    scorer (_sid_predict_proba) and also reports the mean P(AIGC). `soft=False`
    falls back to greedy generate+parse, and `n_votes>1` to majority voting.
    Macro-F1 / balanced accuracy are collapse-aware: a constant predictor scores
    near zero on them even when its 3-class accuracy looks respectable.
    """
    model.eval()
    records = []
    for idx, ex in enumerate(eval_samples):
        gold = int(ex["label"])
        if soft:
            pred, p_aigc, _ = _sid_predict_proba(model, processor, ex["image"])
            decoded = ""
        else:
            pred, decoded = _sid_predict(
                model, processor, ex["image"], max_new_tokens, n_votes, temperature, top_p)
            p_aigc = (1.0 if collapse_to_binary(pred) == "AIGC" else 0.0) if pred is not None else None
        records.append({
            "idx": idx,
            "gold": gold,
            "parsed": pred,
            "p_aigc": p_aigc,
            "decoded": decoded,
            "correct_3class": pred is not None and pred == gold,
            "correct_binary": collapse_to_binary(pred) == collapse_to_binary(gold),
        })
        if (idx + 1) % 10 == 0:
            print(f"Evaluated {idx + 1}/{len(eval_samples)}...")

    details = pd.DataFrame(records)
    acc_3class = float(details["correct_3class"].mean())
    acc_binary = float(details["correct_binary"].mean())
    # Per-class 3-class accuracy so a collapse-to-majority failure is visible.
    per_class = details.groupby("gold")["correct_3class"].agg(["mean", "count"]).to_dict("index")

    # Collapse-aware metrics. Unparsed preds -> sentinel -1 so they count as wrong
    # (and appear as an extra "None" column in the confusion matrix).
    gold_arr = details["gold"].tolist()
    pred_arr = [p if p is not None else -1 for p in details["parsed"]]
    macro_f1 = float(f1_score(gold_arr, pred_arr, labels=[0, 1, 2], average="macro", zero_division=0))
    balanced_acc = float(balanced_accuracy_score(gold_arr, pred_arr))
    cm_labels = [0, 1, 2, -1]
    confusion = confusion_matrix(gold_arr, pred_arr, labels=cm_labels)

    # Threshold-independent binary metrics (parity with SigLIP's table). Needs
    # the soft P(AIGC) score and both binary classes present in the sample.
    auroc = ap = float("nan")
    if soft and details["p_aigc"].notna().all():
        bin_gold = [0 if g == 0 else 1 for g in gold_arr]
        if len(set(bin_gold)) == 2:
            auroc = float(roc_auc_score(bin_gold, details["p_aigc"].tolist()))
            ap = float(average_precision_score(bin_gold, details["p_aigc"].tolist()))

    tag = "soft" if soft else ("greedy" if n_votes == 1 else f"majority@{n_votes}")
    extra = ""
    if soft:
        extra = (f"  meanP(AIGC) {details['p_aigc'].mean():.3f}"
                 f"  AUROC {auroc:.3f}  AP {ap:.3f}")
    print(f"[{tag}] 3-class {acc_3class * 100:.2f}%  binary {acc_binary * 100:.2f}%  "
          f"macroF1 {macro_f1:.3f}  balAcc {balanced_acc * 100:.2f}%{extra}  (n={len(details)})")
    return {
        "acc_3class": acc_3class,
        "acc_binary": acc_binary,
        "macro_f1": macro_f1,
        "balanced_acc": balanced_acc,
        "auroc": auroc,
        "ap": ap,
        "per_class": per_class,
        "confusion": confusion,
        "cm_labels": cm_labels,
        "details": details,
    }

In [ ]:
# ============================================================================
# === BUILD HELD-OUT EVAL SET (content-hash holdout) + BASELINE ACCURACY ===
# ============================================================================
# v2 lever #4. v1 used positional take/skip, which silently leaks if the stream
# ever reshuffles. Here a SID sample is routed to EVAL iff its CONTENT hash
# lands in the holdout bucket; the training cell keeps the complement (same
# rule), so eval and train are disjoint by content -- stable across reshuffles
# AND robust to near-duplicate base photos (a downscaled hash collides them).
import hashlib


def _img_hash(image, size=8):
    """Stable PERCEPTUAL hash (average-hash): downscale to size x size grayscale,
    threshold each pixel at the frame mean to a bit, then md5 the bit pattern.
    Perceptual -> near-duplicate images (a resized / mildly re-encoded copy, or a
    Tampered image sharing a Real base photo) collapse to the SAME digest, so they
    can't straddle the train/eval split; the md5 wrap gives uniform hash%MOD
    bucketing. Returns a hex digest (so int(h, 16) % MOD routes uniformly)."""
    if image is None:
        return None
    g = image.convert("L").resize((size, size))
    px = list(g.getdata())
    avg = sum(px) / len(px)
    bits = "".join("1" if p >= avg else "0" for p in px)
    return hashlib.md5(bits.encode()).hexdigest()


def _is_eval(image):
    """Route ~1/EVAL_HOLDOUT_MOD of images to the held-out eval set, by hash."""
    h = _img_hash(image)
    return h is not None and int(h, 16) % EVAL_HOLDOUT_MOD == 0


# Materialise the eval set: scan the shuffled stream, keep hash-routed samples
# until N_EVAL collected (so baseline and fine-tuned score the SAME images).
eval_samples, _seen = [], 0
for ex in shuffled_stream:
    _seen += 1
    if _is_eval(ex["image"]):
        eval_samples.append(ex)
        if len(eval_samples) >= N_EVAL:
            break
print(f"Held-out eval samples: {len(eval_samples)}  (scanned {_seen} stream rows)")

# BASELINE: evaluate the pre-fine-tuned model. MUST run before sid_trainer.train()
# (below), which mutates `model` in place.
baseline = eval_sid_accuracy(model, processor, eval_samples, soft=SOFT_SCORE)

In [ ]:
# ============================================================================
# === BUILD THE MIXED TRAIN SOURCE: SID_Set stream (+) WildFake subset ===
# v2 lever #2 (generator diversity). Both are datasets.IterableDataset with
# columns {image, label}; interleave_datasets samples between them. Training
# never sees the hash-routed eval samples, nor any WildFake eval image.
# ============================================================================
import os
import csv as _csv
from datasets import Dataset, Image as HFImage, Value, interleave_datasets

# --- SID half: everything the eval holdout did NOT claim (same _is_eval rule) -
sid_train = shuffled_stream.filter(lambda ex: not _is_eval(ex["image"]))
sid_train = sid_train.select_columns(["image", "label"])
sid_train = sid_train.cast_column("image", HFImage()).cast_column("label", Value("int64"))


def _load_wildfake_train():
    """Load wildfake_train/ as an IterableDataset[{image, label}], excluding any
    image whose content hash collides with the wildfake_balanced/ EVAL set.
    WildFake fake images are fully synthetic -> SID label 1 (Synthetic); real->0.
    Returns None if the folder is absent or empty (SID-only fallback)."""
    labels_csv = os.path.join(WILDFAKE_TRAIN_DIR, "labels.csv")
    if not os.path.exists(labels_csv):
        return None

    # Hashes of the eval set -> belt-and-braces anti-leak (in case a file is
    # shared between the train and eval pulls).
    eval_hashes, eval_csv = set(), os.path.join("wildfake_balanced", "labels.csv")
    if os.path.exists(eval_csv):
        with open(eval_csv, newline="") as f:
            for r in _csv.DictReader(f):
                p = os.path.join("wildfake_balanced", r["image_path"])
                if os.path.exists(p):
                    eval_hashes.add(_img_hash(Image.open(p)))

    paths, labels = [], []
    with open(labels_csv, newline="") as f:
        for r in _csv.DictReader(f):
            p = os.path.join(WILDFAKE_TRAIN_DIR, r["image_path"])
            if not os.path.exists(p):
                continue
            if _img_hash(Image.open(p)) in eval_hashes:
                continue
            paths.append(p)
            labels.append(0 if int(r["label"]) == 0 else 1)
    if not paths:
        return None

    ds = (Dataset.from_dict({"image": paths, "label": labels})
          .cast_column("image", HFImage()).cast_column("label", Value("int64")))
    print(f"WildFake train: {len(paths)} images "
          f"(real={labels.count(0)}, fake={labels.count(1)}) after eval-dedup")
    return ds.to_iterable_dataset()


wf_train = _load_wildfake_train() if USE_WILDFAKE_MIX else None
if wf_train is not None:
    train_source = interleave_datasets(
        [sid_train, wf_train],
        probabilities=[1.0 - WILDFAKE_MIX_PROB, WILDFAKE_MIX_PROB],
        seed=SEED, stopping_strategy="all_exhausted",
    )
    print(f"Mixed train source: SID {1 - WILDFAKE_MIX_PROB:.0%} / WildFake {WILDFAKE_MIX_PROB:.0%}")
else:
    train_source = sid_train
    if USE_WILDFAKE_MIX:
        print(f"WARNING: '{WILDFAKE_TRAIN_DIR}/labels.csv' not found -- falling back to "
              f"SID-only. Run pull_wildfake_train.py to enable the generator-diversity mix.")

---
### Augment + map prompt/completion into the mixed stream, then fine-tune

Following the tutorial's pattern (`SFTConfig` / `SFTTrainer`) rather than a
hand-rolled loop, but on the **multimodal** path, over the v2 mixed source:

1. **Disjoint train slice (hash-holdout).** The mixed `train_source` above is the
   complement of the content-hash eval bucket (`_is_eval`), so training never
   sees an eval image — stable across reshuffles (v1 used positional
   `take`/`skip`, which silently leaks if the stream re-shuffles).
2. **Augment, then map prompt/completion into the stream** — each example is
   first passed through `augment_image` (train-time robustness), then gets a
   `"prompt"` (user turn, image placeholder + task) and a `"completion"`
   (assistant `\boxed{d}` answer), keeping the (augmented) `image` column. TRL
   detects the `image` column and uses its native
   `DataCollatorForVisionLanguageModeling`, processing pixels on the fly.
3. Configure `SFTConfig` with `max_length=None` (so image tokens aren't
   truncated), `max_steps=MAX_STEPS` (the source is a streaming `IterableDataset`
   with no length), and **`completion_only_loss=True`** so loss is computed on
   the answer only (the collator masks the prompt to `-100`). Hand the stream to
   `SFTTrainer` with `processing_class=processor` and call `.train()`.

> **Notes.** The image is genuinely fed to the model (vision pathway), and loss
> falls only on the `\boxed{d}` answer — the fix for the earlier
> collapse-to-one-class failure. TRL does not support `assistant_only_loss` for
> vision datasets, hence the prompt/completion + `completion_only_loss` route.
> `OPTIM="adamw_8bit"` requires `bitsandbytes` (installed above).

In [ ]:
# ============================================================================
# === AUGMENT + MAP PROMPT/COMPLETION INTO THE MIXED STREAM (VLM SFT) ===
# ============================================================================
# `train_source` (SID (+) WildFake, built above) is the disjoint complement of
# the hash-routed eval set.
# SPEED (lever #3): ONE lazy .map() instead of two. Augment the image AND attach
# the prompt/completion in a single pass, so the hot streaming path builds one
# generator layer per sample rather than two. TRL's vision collator +
# completion_only_loss still mask the prompt so loss falls only on the \boxed{d}
# answer. With dataloader_num_workers > 0 (set in the trainer cell, lever #1) this
# whole function runs in the worker processes, off the GPU's critical path.
import io  # decode encoded-image dicts (WildFake branch) to PIL


def _as_pil(x):
    """Normalise a sample's image to a PIL.Image. SID samples arrive already
    decoded, but after interleave the WildFake branch (built via from_dict ->
    to_iterable_dataset) can yield the ENCODED {'bytes'|'path'} dict instead --
    the Image feature's decode flag isn't carried through the interleave. Decode
    it here so augment_image (which needs .mode) never sees a dict."""
    if isinstance(x, Image.Image):
        return x
    if isinstance(x, dict):
        if x.get("bytes") is not None:
            return Image.open(io.BytesIO(x["bytes"]))
        if x.get("path"):
            return Image.open(x["path"])
    raise TypeError(f"Unexpected image value of type {type(x)}: {x!r}")


def _prep_example(ex):
    image = _as_pil(ex["image"])
    if AUGMENT_TRAIN:
        image = augment_image(image, _aug_rng)
    return {
        "image": image,
        "prompt": build_sid_train_prompt(),
        "completion": build_sid_train_completion(ex["label"]),
    }


sft_stream = train_source.map(_prep_example, remove_columns=["label"])

# Peek: confirm each sample now carries an (augmented) image + prompt + completion.
_peek = next(iter(sft_stream))
print("keys:", list(_peek.keys()))
print("prompt:    ", _peek["prompt"])
print("completion:", _peek["completion"])
_img = _peek.get("image")
print("image size:", _img.size if _img is not None else None)

In [ ]:
# ============================================================================
# === SET UP SFTTrainer + FINE-TUNE (adapted from the fine-tuning tutorial) ===
# ============================================================================
# NB: run the baseline-eval cell above BEFORE this cell -- .train() mutates
# `model` in place, so the pre-fine-tuned numbers must be captured first.
sid_sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    # Streaming IterableDataset has no length -> drive with max_steps, not epochs.
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    # SPEED: bf16 mixed precision (A100-native) + tf32 matmul kernels. bf16 pairs
    # with the bf16 weights + FlashAttention-2 loaded above; tf32 accelerates the
    # fp32 matmuls that remain. Both are ~free on Ampere and add no memory.
    bf16=True,
    tf32=True,
    optim=OPTIM,               # "adamw_8bit" needs bitsandbytes; else "adamw_torch"
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    seed=SEED,
    report_to="none",
    # SPEED (lever #1): parallelise the input pipeline -- the real fix for the
    # ~2 s/image wall, which was data-loading, not compute. At num_workers=0 the
    # single main process fetches each STREAMED image over the network, augments
    # it (PIL: jpeg/blur/noise/...), and hashes it, all while the A100 waits.
    # Workers prefetch and do that in parallel so the GPU stops starving.
    # CPU-RAM NOTE: RAM scales ~linearly with num_workers -- each worker holds its
    # OWN shuffle buffer (buffer_size=10000, cell 2) PLUS prefetch_factor batches
    # of decoded+augmented full-res PILs. num_workers=8 x prefetch=4 OOM-killed a
    # worker on Colab ("DataLoader worker exited unexpectedly"). 2 x 2 keeps most
    # of the speedup at ~1/4 the RAM. Raise num_workers only if RAM has headroom;
    # if still OOM, drop the shuffle buffer_size (cell 2) or set num_workers=0.
    dataloader_num_workers=2,        # v2c: 4 OOM'd CPU RAM (79/83GB). ~18GB/worker
                                     # from the PIL aug pipeline -> 2 is the ceiling
                                     # on this 83GB box. More speed = GPU-side, not
                                     # more workers (GPU RAM is only 16/40GB used).
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    # VLM specifics: never truncate (would drop image tokens); keep the image
    # column so the vision collator receives it.
    max_length=None,
    remove_unused_columns=False,
    # Fix for the collapse-to-one-class failure: supervise ONLY the \boxed{d}
    # answer, not the identical prompt shared by every example. Without this the
    # full-sequence loss is dominated by the repeated prompt and the one-digit
    # answer signal is swamped -> the model degenerates to a constant class.
    # TRL rejects assistant_only_loss for vision datasets, so we use the
    # prompt/completion format (mapped above) + completion_only_loss: the vision
    # collator sets the prompt-part labels to -100.
    completion_only_loss=True,
)

sid_trainer = SFTTrainer(
    model=model,
    args=sid_sft_config,
    train_dataset=sft_stream,
    eval_dataset=None,
    processing_class=processor,
)

sid_trainer.train()


---
### Results: did fine-tuning help?

Re-run the eval on the fine-tuned `model` and print the side-by-side comparison.
Alongside 3-class / binary accuracy we report **collapse-aware** metrics:

- **macro-F1** and **balanced accuracy** — averaged over the three classes, so a
  model that collapses to one class scores near the floor (~0.15 macro-F1) no
  matter how good its raw 3-class % looks. These are the honest model-selection
  numbers.
- a **3×3 confusion matrix** (rows = gold, cols = pred, plus a `None`/unparsed
  column) — a single populated column is the visual signature of collapse.

We also run **majority voting** (`mv@5`: sample 5 answers, take the mode) on the
fine-tuned model. It reduces variance for a noisy-but-informative model, but
**cannot** rescue a fully collapsed one — so it is reported next to greedy, not
instead of it. A positive Δ binary / Δ macro-F1 means fine-tuning genuinely
helped.


In [ ]:
# ============================================================================
# === FINE-TUNED ACCURACY + BASELINE-VS-FINE-TUNED COMPARISON ===
# ============================================================================
# Same held-out samples, greedy eval on the now-fine-tuned `model`...
finetuned = eval_sid_accuracy(model, processor, eval_samples, soft=SOFT_SCORE)
# ...plus self-consistency: majority vote over several sampled answers. This cuts
# variance for a noisy-but-informative model; it CANNOT rescue a fully collapsed
# one (every vote would be the same class), so read it alongside greedy.
finetuned_mv = eval_sid_accuracy(
    model, processor, eval_samples, EVAL_MAX_NEW_TOKENS,
    n_votes=5, temperature=0.7, top_p=0.9, soft=False,
)


def _row(name, r):
    auroc = r.get("auroc", float("nan"))
    return (f"{name:18s} | 3cls {r['acc_3class'] * 100:5.2f}%  bin {r['acc_binary'] * 100:5.2f}%  "
            f"macroF1 {r['macro_f1']:.3f}  balAcc {r['balanced_acc'] * 100:5.2f}%  "
            f"AUROC {auroc:.3f}")


print("\n=== Baseline vs fine-tuned (macro-F1/balanced-acc are collapse-aware) ===")
print(_row("baseline", baseline))
print(_row("finetuned (greedy)", finetuned))
print(_row("finetuned (mv@5)", finetuned_mv))
print(f"\nΔ binary  (greedy): {(finetuned['acc_binary'] - baseline['acc_binary']) * 100:+.2f} pts")
print(f"Δ macroF1 (greedy): {finetuned['macro_f1'] - baseline['macro_f1']:+.3f}   "
      f"(a constant single-class predictor floors macro-F1 at ~0.15)")

print("\nPer-class 3-class accuracy (gold -> baseline -> finetuned greedy):")
for lbl in sorted(SID_LABELS):
    name = SID_LABELS[lbl][0]
    b = baseline["per_class"].get(lbl, {"mean": float("nan"), "count": 0})
    f = finetuned["per_class"].get(lbl, {"mean": float("nan"), "count": 0})
    print(f"  {lbl} {name:9s} | {b['mean'] * 100:5.1f}%  ->  {f['mean'] * 100:5.1f}%  (n={b['count']})")

print()
print_confusion(baseline["confusion"], baseline["cm_labels"], "Baseline confusion")
print()
print_confusion(finetuned["confusion"], finetuned["cm_labels"], "Fine-tuned (greedy) confusion")


---
### Error Analysis (v2, required deliverable)

Surfaces the fine-tuned model's **most-confident mistakes** on the held-out SID
slice — the authentic images it was surest were AIGC (false positives) and the
AIGC images it was surest were authentic (false negatives) — ranked by the soft
`P(AIGC)` from the pass above. Saves thumbnails + a CSV to `error_analysis/` so
the actual failure images drop straight into the DevPost error-analysis note.
Mirrors SigLIP's error-analysis cell; the `gold_class` column also reveals
whether false negatives cluster on **Tampered** vs **Synthetic**.


In [ ]:
# ============================================================================
# === ERROR ANALYSIS (v2, parity with SigLIP) ================================
# Pull the most-confident misclassifications from the fine-tuned held-out pass
# (finetuned['details'], soft P(AIGC)) and dump thumbnails + a CSV so the real
# failure images can go straight into the DevPost error-analysis note.
#   false positive = authentic (gold 0)  scored AIGC      -> rank by HIGH P(AIGC)
#   false negative = AIGC     (gold 1/2) scored authentic -> rank by LOW  P(AIGC)
# ============================================================================
import os
import pandas as pd

ERROR_ANALYSIS_DIR = os.path.join(OUTPUT_DIR, "error_analysis")
os.makedirs(ERROR_ANALYSIS_DIR, exist_ok=True)
TOP_K = 8

_det = finetuned["details"]          # idx, gold, p_aigc (soft) per eval sample
if _det["p_aigc"].isna().any():
    raise RuntimeError("Error analysis needs soft P(AIGC); re-run eval with SOFT_SCORE=True.")

# (idx, gold, p_aigc) rows, split by binary error at threshold 0.5.
_rows = [(int(r.idx), int(r.gold), float(r.p_aigc)) for r in _det.itertuples(index=False)]
false_positives = sorted([r for r in _rows if r[1] == 0 and r[2] >= 0.5], key=lambda x: -x[2])[:TOP_K]
false_negatives = sorted([r for r in _rows if r[1] != 0 and r[2] < 0.5],  key=lambda x:  x[2])[:TOP_K]

n_fp = sum(1 for r in _rows if r[1] == 0 and r[2] >= 0.5)
n_fn = sum(1 for r in _rows if r[1] != 0 and r[2] < 0.5)
print(f"False positives (authentic -> scored AIGC): {len(false_positives)} shown / {n_fp} total in sample")
print(f"False negatives (AIGC -> scored authentic): {len(false_negatives)} shown / {n_fn} total in sample\n")


def _save_error_set(name, items):
    out = []
    for rank, (idx, gold, p_aigc) in enumerate(items):
        img = eval_samples[idx]["image"]
        cls = SID_LABELS[gold][0] if gold in SID_LABELS else str(gold)
        fname = f"{name}_{rank:02d}_p{p_aigc:.3f}.png"
        img.convert("RGB").save(os.path.join(ERROR_ANALYSIS_DIR, fname))
        out.append({"rank": rank, "eval_index": idx, "gold": gold, "gold_class": cls,
                    "p_aigc": round(p_aigc, 4), "file": fname})
    return pd.DataFrame(out)


fp_df = _save_error_set("false_positive", false_positives)
fn_df = _save_error_set("false_negative", false_negatives)

print("Top false positives (authentic -> scored AIGC):")
try:
    display(fp_df)
except NameError:
    print(fp_df.to_string(index=False))
print("\nTop false negatives (AIGC -> scored authentic):")
try:
    display(fn_df)
except NameError:
    print(fn_df.to_string(index=False))

pd.concat([fp_df.assign(type="false_positive"), fn_df.assign(type="false_negative")]).to_csv(
    os.path.join(ERROR_ANALYSIS_DIR, "error_analysis.csv"), index=False)
print(f"\nThumbnails + CSV saved to {ERROR_ANALYSIS_DIR}. Inspect them and write 2-3 sentences "
      f"per failure mode for the DevPost note (e.g. do false negatives cluster on Tampered vs "
      f"Synthetic, or on a particular texture / compression level?).")


---
### WildFake validation benchmark (reference-only)

The brief supplies a **demo subset of WildFake** to track progress — it is *not*
scored and **must never be trained on**:

| bucket | source | count |
| :--- | :--- | ---: |
| Non-AIGC (authentic) | COCO val2017 | 4998 |
| AIGC | DALL·E Advanced | 8843 |

We pull it from ModelScope (`hy2628982280/WildFake`) **streamed**
(`use_streaming=True`) — same disk-light approach as the SID_Set training
stream. The repo hosts the *full* WildFake, so the cell below **filters** to the
two demo buckets using the label columns confirmed from the dataset's CSVs:

- **`IsFake`** — the real/fake flag (`0` real, `1` AI-generated).
- **`Architecture` / `Image_path`** — source; `real_coco` rows carry `coco`,
  DALL·E rows carry `DALLE`.
- **`IsAdvanced`** — marks the "Advanced" split, so DALL·E Advanced =
  `IsFake=1 & DALLE & IsAdvanced=1`.

Metric is **binary only** (AIGC vs authentic) — the deliverable target. Each
scored image yields a `pred` AIGC likelihood (`1.0`/`0.0`) plus its
`image_path`, mirroring the final JSON output format. The model's 3-class
`\boxed{d}` prediction is collapsed to binary via `collapse_to_binary` from the
eval harness above.

> The cell first **peeks one streamed row** and prints its keys. Confirm the row
> exposes the expected columns and how it carries pixels (a decoded image vs a
> bare `Image_path`); if it's path-only, set `WF_IMAGES_ROOT` to the downloaded
> Images root so paths resolve. Then uncomment the final line to run.


In [ ]:
# ============================================================================
# === WILDFAKE VALIDATION BENCHMARK (reference-only, streamed, BINARY) ===
# COCO val2017 non-AIGC + DALL·E Advanced AIGC -- the brief's demo subset.
# NEVER trained on: this is a held-out reference benchmark (problem brief).
# Reuses the model's prediction path (_sid_predict) + collapse_to_binary above.
# ============================================================================
from modelscope.msdatasets import MsDataset
from PIL import Image
import os

# --- Schema confirmed from the dataset's label CSVs -------------------------
# Columns: Generator, Architecture, Weight, Category, IsAdvanced, IsFake,
#          Image_path, Num.  Example DALL-E-3 row:
#   Generator=Diffusion_based  Architecture=DALLE  IsAdvanced=1  IsFake=1
#   Image_path=./Diffusion_based/DALLE/Advanced/DALLE3/.../xxx.jpg
# => IsFake is the real/fake flag; Architecture=="DALLE" & IsAdvanced==1 is the
#    "DALL-E Advanced" AIGC bucket; COCO reals sit in real_coco (path has "coco").
# Values may arrive as ints or strings ("1"/"0"/"True") -> coerce defensively.

WF_DATASET = "hy2628982280/WildFake"
WF_SUBSET  = "default"
WF_SPLIT   = "train"
# Cap per class for a quick pass; raise toward the full 4998 / 8843 for the
# official demo numbers (None = take every matching row).
WF_MAX_AUTHENTIC = 200
WF_MAX_AIGC      = 200
# Streamed rows may carry only Image_path (a relative "./..."), not decoded
# pixels. If so, point this at the local Images root so paths resolve; the peek
# below reveals which case we're in. None = assume the row already has an image.
WF_IMAGES_ROOT = None


def _wf_truthy(v):
    """Coerce IsFake / IsAdvanced ('1'/'0'/1/0/'True'/'False') to bool."""
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y", "t"}
    return bool(v)


def _wf_haystack(ex):
    """Lower-cased blob of the source-identifying fields, for substring tests."""
    return " ".join(
        str(ex.get(k, "")) for k in ("Architecture", "Category", "Generator", "Image_path")
    ).lower()


def _wf_is_authentic_coco(ex):
    """Non-AIGC COCO real image: not fake AND source mentions coco."""
    return (not _wf_truthy(ex.get("IsFake"))) and "coco" in _wf_haystack(ex)


def _wf_is_dalle_advanced(ex):
    """AIGC DALL-E Advanced: fake AND DALLE source AND IsAdvanced."""
    return (
        _wf_truthy(ex.get("IsFake"))
        and "dalle" in _wf_haystack(ex)
        and _wf_truthy(ex.get("IsAdvanced"))
    )


def _wf_image(ex):
    """Return a PIL.Image (RGB) for a streamed row, or None if unresolvable.

    Prefers a decoded image the loader already provides; else opens Image_path
    (joined under WF_IMAGES_ROOT when the path is relative).
    """
    for k in ("image", "Image", "img"):
        obj = ex.get(k)
        if isinstance(obj, Image.Image):
            return obj.convert("RGB")
    path = ex.get("Image_path")
    if path:
        if WF_IMAGES_ROOT:
            path = os.path.join(WF_IMAGES_ROOT, str(path).lstrip("./"))
        try:
            return Image.open(path).convert("RGB")
        except Exception:
            return None
    return None


# --- Peek one streamed row so the ACTUAL keys are visible on Colab -----------
# (schema above is from the label CSVs; confirm the streamed row exposes the
#  same keys -- and whether it carries decoded pixels or just Image_path.)
_wf_peek_stream = MsDataset.load(
    WF_DATASET, subset_name=WF_SUBSET, split=WF_SPLIT, use_streaming=True,
)
_wf_peek = next(iter(_wf_peek_stream))
print("WildFake row keys:", list(_wf_peek.keys()))
print("sample row:", {k: _wf_peek[k] for k in list(_wf_peek)[:8]})


@torch.no_grad()
def eval_wildfake_binary(model, processor,
                         max_authentic=WF_MAX_AUTHENTIC, max_aigc=WF_MAX_AIGC,
                         max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Binary AIGC-vs-authentic eval over the streamed WildFake demo subset.

    Streams the split fresh, keeps up to `max_authentic` COCO reals + `max_aigc`
    DALL-E-Advanced fakes, greedy-predicts each (3-class \\boxed{d}), collapses
    to binary, and reports accuracy + a per-class breakdown. Each record's `pred`
    is the model's AIGC likelihood (1.0 = AIGC, 0.0 = authentic) -- the field the
    final deliverable JSON needs, alongside `image_path`.
    """
    model.eval()
    stream = MsDataset.load(
        WF_DATASET, subset_name=WF_SUBSET, split=WF_SPLIT, use_streaming=True,
    )
    want = {"authentic": max_authentic, "AIGC": max_aigc}
    got  = {"authentic": 0, "AIGC": 0}

    def _full(c):
        return want[c] is not None and got[c] >= want[c]

    records = []
    for ex in stream:
        if _wf_is_authentic_coco(ex):
            gold = "authentic"
        elif _wf_is_dalle_advanced(ex):
            gold = "AIGC"
        else:
            continue
        if _full(gold):
            if _full("authentic") and _full("AIGC"):
                break
            continue                       # this class done; keep filling the other
        img = _wf_image(ex)
        if img is None:                    # pixels unresolved -> skip (see WF_IMAGES_ROOT)
            continue
        label3, _ = _sid_predict(model, processor, img, max_new_tokens, 1, 0.7, 0.9)
        pred_bin = collapse_to_binary(label3)
        got[gold] += 1
        records.append({
            "image_path": ex.get("Image_path"),
            "gold": gold,
            "pred": 1.0 if pred_bin == "AIGC" else 0.0,   # AIGC likelihood (deliverable field)
            "pred_bin": pred_bin,
            "correct": pred_bin == gold,
        })
        if len(records) % 20 == 0:
            print(f"scored {len(records)}  (authentic {got['authentic']}, AIGC {got['AIGC']})")

    details = pd.DataFrame(records)
    if len(details) == 0:
        print("No samples scored -- check the peek above (keys / how pixels are "
              "exposed) and set WF_IMAGES_ROOT if rows carry only Image_path.")
        return {"acc_binary": float("nan"), "per_class": {}, "details": details}

    acc = float(details["correct"].mean())
    per_class = details.groupby("gold")["correct"].agg(["mean", "count"]).to_dict("index")
    print(f"\nWildFake binary accuracy: {acc * 100:.2f}%  (n={len(details)})")
    for cls in ("authentic", "AIGC"):
        c = per_class.get(cls, {"mean": float("nan"), "count": 0})
        print(f"  {cls:9s}: {c['mean'] * 100:5.1f}%  (n={c['count']})")
    return {"acc_binary": acc, "per_class": per_class, "details": details}


# Uncomment to run once the peek confirms keys / pixel access:
# wf_result = eval_wildfake_binary(model, processor)


---
### Local WildFake test-set eval (preferred over streaming)

WildFake stores its pixels **inside multi-GB per-category zips**, keyed by
`Image_path` — so `MsDataset.load(streaming=True)` (the cell above) tends to
hand back path strings, not decoded images. The reliable path is to pull a small
test set to disk with `pull_wildfake_balanced.py` (HTTP range requests, no full
download) and score that folder directly. In v2 this `wildfake_balanced/` set is
the **eval** half only — the training WildFake images live in `wildfake_train/`
(a separate, disjoint pull), so scoring it here never touches trained data.

This cell reads `wildfake_balanced/labels.csv`
(`image_path, category, generator, label`, with **label 0 = real / 1 = fake**),
loads each image, runs the soft `_sid_predict_proba` → threshold-0.5
prediction path (v2), and reports:

- **overall binary accuracy** (AIGC vs authentic — the deliverable metric),
- **per-class** accuracy (authentic vs AIGC — a collapse flattens one to ~0),
- **per-category** accuracy, worst-first (which *generators* fool the model;
  the set is category-balanced so these compare directly).

It also writes **`wildfake_balanced_v2_preds.json`** in the deliverable
shape (`[{image_path, pred}]`, `pred` = the **soft** AIGC likelihood). Set `TEST_DIR` if your folder
name differs; the Colab bootstrap cell `%cd`s into `TechJam`, so the default
resolves there. Reference-only — **never trained on**.


In [ ]:
# ============================================================================
# === LOCAL WILDFAKE TEST-SET EVAL (labeled folder + labels.csv, BINARY) ===
# Scores a locally-pulled WildFake test set (from pull_wildfake_balanced.py): a
# folder of category subdirs + labels.csv (image_path relative to the folder,
# label 0=real / 1=fake). Disk-light + reproducible -- preferred over MsDataset
# streaming, since WildFake stores pixels inside multi-GB zips, not decoded
# stream rows. Reuses _sid_predict + collapse_to_binary from the eval harness.
# NEVER trained on: reference-only test set (problem brief).
# ============================================================================
import os
import json
from PIL import Image

# TEST_DIR is relative to the working dir. The Colab bootstrap cell %cd's into
# TechJam, so "wildfake_balanced" resolves there (and locally too).
TEST_DIR   = "wildfake_balanced"
LABELS_CSV = os.path.join(TEST_DIR, "labels.csv")
OUT_JSON   = "wildfake_balanced_v2_preds.json"   # deliverable-shaped: [{image_path, pred}]


def _bin_from_label(label) -> str:
    """WildFake test label -> binary: 0 -> authentic, anything else -> AIGC.
    Matches collapse_to_binary's real/AIGC split so gold and pred are comparable."""
    return "authentic" if int(label) == 0 else "AIGC"


@torch.no_grad()
def eval_local_folder(model, processor, test_dir=TEST_DIR, labels_csv=LABELS_CSV,
                      max_new_tokens=EVAL_MAX_NEW_TOKENS, out_json=OUT_JSON):
    """Binary AIGC-vs-authentic eval over a local labeled image folder.

    Reads labels_csv (columns include `image_path` relative to test_dir and a
    `label` 0=real / 1=fake), greedy-predicts each image (3-class \\boxed{d}),
    collapses to binary, and reports overall / per-class / per-category accuracy.
    Writes deliverable-shaped predictions ([{image_path, pred}]) to out_json.
    """
    model.eval()
    df = pd.read_csv(labels_csv)
    print(f"Loaded {len(df)} labeled rows from {labels_csv}")

    records = []
    for i, row in enumerate(df.itertuples(index=False)):
        gold = _bin_from_label(row.label)
        cat = getattr(row, "category", "")
        path = os.path.join(test_dir, row.image_path)
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"skip (can't open) {path}: {e}")
            continue
        if SOFT_SCORE:
            label3, p_aigc, _ = _sid_predict_proba(model, processor, img)
        else:
            label3, _ = _sid_predict(model, processor, img, max_new_tokens, 1, 0.7, 0.9)
            p_aigc = 1.0 if collapse_to_binary(label3) == "AIGC" else 0.0
        pred_bin = "AIGC" if p_aigc >= 0.5 else "authentic"
        records.append({
            "image_path": path,
            "category": cat,
            "gold": gold,
            "pred": float(p_aigc),          # SOFT AIGC likelihood (deliverable field)
            "pred_bin": pred_bin,
            "parsed": label3,
            "correct": pred_bin == gold,
        })
        if (i + 1) % 20 == 0:
            print(f"scored {i + 1}/{len(df)}...")

    details = pd.DataFrame(records)
    acc = float(details["correct"].mean())
    n_unparsed = int(details["parsed"].isna().sum())

    # Deliverable JSON: only image_path + pred, one row per scored image.
    with open(out_json, "w") as f:
        json.dump(
            [{"image_path": r["image_path"], "pred": r["pred"]} for r in records],
            f, indent=2,
        )

    print(f"\nLocal test-set binary accuracy: {acc * 100:.2f}%  "
          f"(n={len(details)}, unparsed={n_unparsed})")
    # Per-class (authentic vs AIGC): a collapse would flatten one of these to ~0.
    for cls in ("authentic", "AIGC"):
        sub = details[details["gold"] == cls]
        if len(sub):
            print(f"  {cls:9s}: {sub['correct'].mean() * 100:5.1f}%  (n={len(sub)})")
    # Per-category (which generators fool the model). The set is category-
    # balanced, so these are directly comparable; sorted worst-first.
    if "category" in details and details["category"].astype(bool).any():
        print("\nPer-category accuracy (worst-first):")
        by_cat = details.groupby("category")["correct"].agg(["mean", "count"])
        for cat, r in by_cat.sort_values("mean").iterrows():
            print(f"  {cat:<16} {r['mean'] * 100:5.1f}%  (n={int(r['count'])})")
    print(f"\nWrote deliverable predictions -> {out_json}")
    return {"acc_binary": acc, "details": details}


# Uncomment to run:
# local_result = eval_local_folder(model, processor)


In [ ]:
# ============================================================================
# === WILDFAKE LOCAL TEST-SET ACCURACY + CROSS-DATASET COMPARISON ===
# ============================================================================
# Score the local WildFake test set (soft, thresh 0.5). eval_local_folder prints
# overall / per-class / per-category accuracy and writes wildfake_balanced_v2_preds.json.
wildfake = eval_local_folder(model, processor)

# Context: WildFake is a CROSS-DATASET test -- different generators + reals than
# SID_Set -- so compare its binary accuracy against the in-distribution SID_Set
# fine-tuned binary from the comparison cell above. A large drop = source shift.
print("\n=== In-distribution (SID_Set) vs cross-dataset (WildFake) -- binary ===")
if "finetuned" in globals():
    sid_bin = finetuned["acc_binary"]
    wf_bin = wildfake["acc_binary"]
    print(f"  SID_Set held-out (fine-tuned, greedy) : {sid_bin * 100:5.2f}%")
    print(f"  WildFake local test set (greedy)      : {wf_bin * 100:5.2f}%")
    print(f"  Δ (WildFake - SID_Set)                : {(wf_bin - sid_bin) * 100:+.2f} pts")
else:
    print("  (run the fine-tuned comparison cell above first for the SID_Set number)")


---
### Robustness re-eval (v2) — same grid v1 was scored on

Re-measure v2 under the exact transform × severity grid that produced
`robustness_summary.csv` for v1, on the **same** WildFake balanced set
(`wildfake_balanced/`, reference-only). Uses the deterministic `_t_*` transforms
and the soft `_sid_predict_proba` (threshold 0.5) and writes
`robustness_summary_v2.csv` (v1's columns **plus** threshold-independent `AUROC`/`AP`,
so v1 vs v2 still compare directly) alongside `by_source_summary_v2.csv` — a
per-generator clean breakdown mirroring SigLIP's cross-source generalization check.

Watch the **noise** rows: v1's `Acc_AIGC` fell to ~0.22 there; if augmentation
worked, v2 should hold that column up (and the jpeg/blur/resize "gains" should
shrink toward small honest drops as the authentic-bias is corrected).

In [ ]:
# ============================================================================
# === ROBUSTNESS RE-EVAL (v2): transform x severity grid -> summary CSV ===
# Same grid + same WildFake balanced set v1 was scored on, so the two CSVs are
# directly comparable. Deterministic transforms + soft prediction (thresh 0.5).
# v2 parity add-ons (SigLIP): AUROC/AP per row (threshold-independent) + a
# per-generator clean breakdown (cross-source generalization). Reference-only
# set -- never trained on.
# ============================================================================
import os
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score

ROBUST_TEST_DIR   = "wildfake_balanced"        # the reference eval set
ROBUST_OUT_CSV    = "robustness_summary_v2.csv"
BY_SOURCE_OUT_CSV = "by_source_summary_v2.csv"

# (transform, severity) grid -- matches robustness_summary.csv / results.csv exactly.
ROBUST_GRID = [
    ("clean", None),
    ("jpeg", 90), ("jpeg", 70), ("jpeg", 50), ("jpeg", 30),
    ("blur", 0.5), ("blur", 1.0), ("blur", 2.0),
    ("resize", 0.5), ("resize", 0.25),
    ("noise", 0.02), ("noise", 0.05), ("noise", 0.1),
    ("color", 1.2),
    ("crop", 0.8),
]


def _apply_transform(name, sev, img):
    if name == "clean":  return img
    if name == "jpeg":   return _t_jpeg(img, sev)
    if name == "blur":   return _t_blur(img, sev)
    if name == "resize": return _t_resize(img, sev)
    if name == "noise":  return _t_noise(img, sev)
    if name == "color":  return _t_color(img, sev)
    if name == "crop":   return _t_crop(img, sev)
    raise ValueError(name)


def _safe_auroc_ap(gold_bin, scores):
    """AUROC/AP if both binary classes are present, else nan (single-class slice)."""
    if len(set(gold_bin)) < 2:
        return float("nan"), float("nan")
    return (float(roc_auc_score(gold_bin, scores)),
            float(average_precision_score(gold_bin, scores)))


@torch.no_grad()
def eval_robustness_v2(model, processor, test_dir=ROBUST_TEST_DIR,
                       out_csv=ROBUST_OUT_CSV, by_source_csv=BY_SOURCE_OUT_CSV):
    model.eval()
    df = pd.read_csv(os.path.join(test_dir, "labels.csv"))
    # Load every image once (clean) with its binary gold + source generator;
    # transforms are applied per row below.
    base = []
    for row in df.itertuples(index=False):
        try:
            img = Image.open(os.path.join(test_dir, row.image_path)).convert("RGB")
        except Exception:
            continue
        gold_bin = 0 if int(row.label) == 0 else 1
        base.append((img, "authentic" if gold_bin == 0 else "AIGC", gold_bin, str(row.generator)))
    print(f"Robustness set: {len(base)} images")

    rows, clean_acc, clean_records = [], None, None
    for name, sev in ROBUST_GRID:
        n = n_auth = n_aigc = c = c_auth = c_aigc = 0
        scores, golds, records = [], [], []
        for img, gold, gold_bin, source in base:
            _, p_aigc, _ = _sid_predict_proba(model, processor, _apply_transform(name, sev, img))
            pred = "AIGC" if p_aigc >= 0.5 else "authentic"
            ok = int(pred == gold)
            n += 1; c += ok
            scores.append(p_aigc); golds.append(gold_bin); records.append((source, ok))
            if gold == "authentic": n_auth += 1; c_auth += ok
            else:                   n_aigc += 1; c_aigc += ok
        acc = c / n if n else float("nan")
        auroc, ap = _safe_auroc_ap(golds, scores)
        if name == "clean":
            clean_acc, clean_records = acc, records   # keep for by-source breakdown
        rows.append({
            "transform": name,
            "severity": "-" if sev is None else sev,
            "Accuracy": acc,
            "Acc_authentic": c_auth / n_auth if n_auth else float("nan"),
            "Acc_AIGC": c_aigc / n_aigc if n_aigc else float("nan"),
            "Acc_drop_vs_clean": (clean_acc - acc) if clean_acc is not None else 0.0,
            "AUROC": auroc,
            "AP": ap,
        })
        print(f"  {name:6s} {str(sev):>5} | acc {acc*100:5.1f}%  "
              f"auth {rows[-1]['Acc_authentic']*100:5.1f}%  aigc {rows[-1]['Acc_AIGC']*100:5.1f}%  "
              f"AUROC {auroc:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_csv, index=False)
    print(f"\nWrote {out_csv}")

    # --- Per-generator clean breakdown (cross-source generalization check) ----
    by_src = {}
    for source, ok in clean_records:
        d = by_src.setdefault(source, {"n": 0, "correct": 0})
        d["n"] += 1; d["correct"] += ok
    by_source = pd.DataFrame(
        [{"source": s, "n": d["n"], "Accuracy": d["correct"] / d["n"]}
         for s, d in sorted(by_src.items())]
    )
    by_source.to_csv(by_source_csv, index=False)
    print(f"Wrote {by_source_csv}  (clean accuracy per generator / source)")

    try:
        print("\nRobustness summary:"); display(summary.round(4))
        print("Clean accuracy by source:"); display(by_source.round(4))
    except NameError:
        print("\nRobustness summary:\n" + summary.round(4).to_string(index=False))
        print("\nClean accuracy by source:\n" + by_source.round(4).to_string(index=False))
    return summary, by_source


# Run it (requires the fine-tuned `model`; reference-only set, never trained on).
robustness_v2, by_source_v2 = eval_robustness_v2(model, processor)


---
### Save / reload the fine-tuned model

Colab's local disk is wiped when the runtime ends, so persist the fine-tuned
weights to the **Drive-mounted `TechJam` folder** before closing. `save_pretrained`
writes a self-contained folder (`config.json` + safetensors weights + processor /
tokenizer) that `from_pretrained` can reload — for submission or for scoring other
test sets.

- **Save cell** — writes `aigc_detector_qwen/` to Drive (survives the runtime
  closing). Uncomment the `shutil.make_archive` line to also get a single
  `.zip` for submission/download.
- **Reload cell** — rebuilds `model` / `processor` / `tokenizer` in a **fresh**
  session (run the pip-install + Drive-mount bootstrap first). Every eval
  function above then works unchanged.

> This saves the merged full weights (a plain `save_pretrained`), not just a
> LoRA adapter — we fine-tuned all params, so the folder is a complete model.


In [ ]:
# ============================================================================
# === SAVE FINE-TUNED MODEL + PROCESSOR (persist before the runtime closes) ===
# ============================================================================
# Colab wipes local disk when the runtime ends. Save to the Drive-mounted TechJam
# folder (cwd after the bootstrap %cd) so the weights survive and can be reloaded
# or submitted. save_pretrained writes config + weights (safetensors) + the
# processor / tokenizer files -- everything from_pretrained needs to reload.
import os

SAVE_DIR = "aigc_detector_qwen_v2"   # relative to TechJam on Drive -> persists

model.save_pretrained(SAVE_DIR, safe_serialization=True)   # config.json + *.safetensors
processor.save_pretrained(SAVE_DIR)                         # processor + tokenizer files

files = sorted(os.listdir(SAVE_DIR))
size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, f))
              for f in files if os.path.isfile(os.path.join(SAVE_DIR, f))) / 1e6
print(f"Saved fine-tuned model -> {os.path.abspath(SAVE_DIR)}  ({size_mb:.1f} MB)")
print("files:", files)

# Optional: bundle into one zip for submission / download.
# import shutil; shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
# print("zipped ->", SAVE_DIR + ".zip")


In [ ]:
# ============================================================================
# === RELOAD THE SAVED MODEL (fresh runtime / other test sets) ===
# ============================================================================
# Run this in a NEW session AFTER the pip-install + Drive-mount bootstrap cells.
# It rebuilds `model`, `processor`, `tokenizer` from the saved folder so the eval
# functions (eval_local_folder / eval_sid_accuracy / eval_wildfake_binary) work
# unchanged. NB: in the CURRENT session the model is already loaded -- running
# this reloads a SECOND copy onto the GPU (extra memory), so it's mainly meant
# for a fresh runtime.
from transformers import AutoModelForImageTextToText, AutoProcessor

SAVE_DIR = "aigc_detector_qwen_v2"
processor = AutoProcessor.from_pretrained(SAVE_DIR)
model = AutoModelForImageTextToText.from_pretrained(
    SAVE_DIR, device_map="auto", dtype="auto",
)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:            # match the original load-cell setup
    tokenizer.pad_token = tokenizer.eos_token
print("Reloaded model + processor from", SAVE_DIR)
